Задание 1. Определить студента/студентов с максимальным средним баллом.

Задание 2. Вычислить средний балл по каждому предмету.

Задание 3. По каждому предмету определить группу с лучшим средним баллом.

In [10]:
// Потенциально необходимые модули

using Newtonsoft.Json;
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;


// Оглащение класса даты для инкапсуляции данных o студенте

public class Data
{
    public string name { get; set; }
    public string group { get; set; }
    public string discipline { get; set; }
    public int mark { get; set; }
}


// Оглащение класса для инкапсуляции данных из файла

public class Exam_Statement
{
    public string taskName { get; set; }
    public Data[] data { get; set; }
}


// Оглашение класса для форматирования выходного файла перед созданием

public class File_Out
{
    public List<Dictionary<string, object>> Response { get; set; }
    public File_Out(List<Dictionary<string, object>> a)
    {
        Response = a;
    }
}

In [11]:
// Метод обработки файла c возвратом форматированного файла для первого задания

public File_Out GetStudentsWithHighestGPA(Data[] file_in)
{
    var studentGpas = file_in.GroupBy(s => s.name).Select(g => new { name = g.Key, GPA = g.Average(s => s.mark) });
    var maxGpa = studentGpas.Max(s => s.GPA);
    var cadets = studentGpas.Where(s => s.GPA == maxGpa).Select(s => s.name).ToArray();
    return new File_Out(cadets.Select(c => new Dictionary<string, object>() { { "Cadet", c }, { "GPA", maxGpa } }).ToList());
}


// Метод обработки файла c возвратом форматированного файла для второго задания

public File_Out CalculateGPAByDiscipline(Data[] file_in)
{
    var disciplinesGpas = file_in.GroupBy(s => s.discipline).Select(g => new { discipline = g.Key, GPA = g.Average(s => s.mark) }).ToDictionary(a => a.discipline, a => a.GPA);
    return new File_Out(disciplinesGpas.Select(kvp => new Dictionary<string, object>() { { kvp.Key, kvp.Value } }).ToList());
}


// Метод обработки файла c возвратом форматированного файла для третьего задания

public File_Out GetBestGroupsByDiscipline(Data[] file_in)
{
    var disciplineGroups = file_in.GroupBy(s => s.discipline).Select(g => new { Discipline = g.Key, Groups = g.GroupBy(s => s.group).Select(gg => new { Group = gg.Key, GPA = gg.Average(s => s.mark) }) });
    var bestGroups = disciplineGroups.Select(dg => (Discipline: dg.Discipline, Group: dg.Groups.OrderByDescending(g => g.GPA).First().Group, GPA: dg.Groups.OrderByDescending(g => g.GPA).First().GPA)).ToArray();
    return new File_Out(bestGroups.Select(i => new Dictionary<string, object>() { { "Discipline", i.Discipline }, { "Group", i.Group }, { "GPA", i.GPA } }).ToList());
}

In [12]:
// Код программы, определяющей тип задания и выполняющей соответствующую функцию c возвратом выходного файла

string[] fileNames = { "GetStudentsWithHighestGPA.json", "CalculateGPAByDiscipline.json", "GetBestGroupsByDiscipline.json" };
foreach (string name in fileNames)
{
    var file = JsonConvert.DeserializeObject<Exam_Statement>(File.ReadAllText(name));
    switch (file.taskName)
    {
        case "GetStudentsWithHighestGPA":
            File.WriteAllText("Output_GetStudentsWithHighestGPA.json", JsonConvert.SerializeObject(GetStudentsWithHighestGPA(file.data), Formatting.Indented));
            break;
        case "CalculateGPAByDiscipline":
            File.WriteAllText("Output_CalculateGPAByDiscipline.json", JsonConvert.SerializeObject(CalculateGPAByDiscipline(file.data), Formatting.Indented));
            break;
        case "GetBestGroupsByDiscipline":
            File.WriteAllText("Output_GetBestGroupsByDiscipline.json", JsonConvert.SerializeObject(GetBestGroupsByDiscipline(file.data), Formatting.Indented));
            break;
        default:
            Console.WriteLine("Неизвестная задача.");
            break;
    }
}